# 09 - Stylometric features on MAGE

07 checked the 14 features on HC3. This checks the same features on MAGE, where the reported results come from (§3.2).

Computed on Viking by `scripts/run_stylometric_features.py`; this notebook loads that file.

1. Do the features tell human and machine text apart, as they did on HC3?
2. If not, is that because the generators pull in opposite directions?
3. Are the leave-one-family-out folds confounded with domain?

## 1. Load the features

Computed once over the whole corpus, so 10 and 12 read the same file.

In [1]:
import numpy as np
import pandas as pd

from logos.paths import DATA_DIR
from logos.features.stylometric import FEATURE_NAMES
from logos.evaluation.metrics import cohens_d

MAGE_DIR = DATA_DIR / "mage"
LABELS = {0: "human", 1: "machine"}

features = pd.read_parquet(MAGE_DIR / "features" / "stylometric.parquet")
corpus = pd.read_parquet(
    MAGE_DIR / "corpus.parquet",
    columns=["text_id", "label", "domain", "family", "model", "split", "n_words"],
)

features = features.merge(corpus, on="text_id")

print("features:", features.shape)
print(features["label"].map(LABELS).value_counts().to_dict())
print("families:", features.loc[features["label"] == 1, "family"].nunique())
print("domains:", features["domain"].nunique())

features: (392385, 21)
{'machine': 243160, 'human': 149225}
families: 7
domains: 10


In [2]:
features[FEATURE_NAMES].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
sentence_length_mean,392385.0,19.483,14.399,1.229,13.158,18.000,23.000,1108.000
sentence_length_sd,392385.0,8.419,11.512,0.000,4.320,6.947,10.008,506.500
mattr,392385.0,0.885,0.071,0.076,0.855,0.895,0.931,1.000
hapax_rate,392385.0,0.759,0.141,0.000,0.700,0.784,0.852,1.000
stopword_rate,392385.0,0.452,0.078,0.000,0.404,0.457,0.505,0.964
punctuation_rate,392385.0,0.029,0.015,0.000,0.021,0.026,0.034,0.988
noun_rate,392385.0,0.280,0.086,0.000,0.218,0.273,0.333,1.000
verb_rate,392385.0,0.184,0.047,0.000,0.155,0.185,0.213,0.625
past_tense_rate,392385.0,0.063,0.048,0.000,0.027,0.054,0.094,0.477
adverb_rate,392385.0,0.057,0.037,0.000,0.030,0.054,0.079,0.533


## 2. Do the features tell the classes apart?

Cohen's d (Cohen, 1988), defined in Appendix D.

The two sets are not measured on comparable text: HC3's human writing is answers to questions, MAGE's spans ten kinds. Cohen's d divides the gap between the classes by how much the texts vary among themselves, so the more varied MAGE's human writing is, the smaller d becomes for the same gap. Section 4 tests how much of the fall that explains.

In [3]:
hc3 = pd.read_parquet(DATA_DIR / "hc3" / "features" / "stylometric.parquet")

mage_labels = features["label"].to_numpy()
hc3_labels = hc3["label"].to_numpy()

separation = pd.DataFrame({
    "d_hc3": [cohens_d(hc3[f].to_numpy(), hc3_labels) for f in FEATURE_NAMES],
    "d_mage": [cohens_d(features[f].to_numpy(), mage_labels) for f in FEATURE_NAMES],
}, index=FEATURE_NAMES)

separation["change"] = separation["d_mage"] - separation["d_hc3"]

print("features reaching d >= 0.5")
print("on HC3: ", (separation["d_hc3"].abs() >= 0.5).sum(), "of 14")
print("on MAGE:", (separation["d_mage"].abs() >= 0.5).sum(), "of 14")

separation.sort_values("d_hc3", key=abs, ascending=False).round(3)

features reaching d >= 0.5
on HC3:  6 of 14
on MAGE: 0 of 14


,d_hc3,d_mage,change
hapax_rate,-1.264,-0.028,1.236
flesch_reading_ease,-0.812,-0.149,0.663
sentence_length_mean,0.801,0.165,-0.636
adverb_rate,-0.776,-0.068,0.708
punctuation_rate,-0.685,-0.091,0.595
contraction_rate,-0.547,0.088,0.635
pronoun_rate,-0.489,0.021,0.511
first_person_rate,-0.440,-0.071,0.369
sentence_length_sd,0.418,0.133,-0.285
mattr,-0.341,0.009,0.350


The 14 features were chosen on HC3, so the order matters more than the values. The two rankings are compared with Spearman's rank correlation (Spearman, 1904). If the ones that worked best there are still best here, that choice holds beyond ChatGPT. If the order has changed, they were selected on a corpus that did not represent the conditions they are reported under.

Reselecting them on MAGE is not an option. Choosing features by looking at MAGE would leak the evaluation corpus into the design, and the reported results would stop being out of sample. §3.6 fixes every design decision before MAGE is touched, so a failure here is reported rather than corrected.

In [4]:
ranks = pd.DataFrame({
    "rank_hc3": separation["d_hc3"].abs().rank(ascending=False).astype(int),
    "rank_mage": separation["d_mage"].abs().rank(ascending=False).astype(int),
})
ranks["moved"] = ranks["rank_hc3"] - ranks["rank_mage"]

correlation = separation["d_hc3"].abs().corr(separation["d_mage"].abs(), method="spearman")
print(f"Spearman correlation of the two rankings: {correlation:.3f}")

ranks.sort_values("rank_hc3")

Spearman correlation of the two rankings: 0.121


,rank_hc3,rank_mage,moved
hapax_rate,1,12,-11
flesch_reading_ease,2,2,0
sentence_length_mean,3,1,2
adverb_rate,4,9,-5
punctuation_rate,5,6,-1
contraction_rate,6,7,-1
pronoun_rate,7,13,-6
first_person_rate,8,8,0
sentence_length_sd,9,4,5
mattr,10,14,-4


A d near zero has two explanations: the feature carries no signal, or it carries signal in opposite directions for different generators and they cancel. Measuring each family against the same human pool separates the two.

In [5]:
human_rows = features[features["label"] == 0]


def family_d(group):
    out = []
    for f in FEATURE_NAMES:
        values = np.concatenate([human_rows[f].to_numpy(), group[f].to_numpy()])
        labels = np.concatenate([np.zeros(len(human_rows)), np.ones(len(group))])
        out.append(cohens_d(values, labels))
    return pd.Series(out, index=FEATURE_NAMES)


machine_rows = features[features["label"] == 1]
per_family = machine_rows.groupby("family")[FEATURE_NAMES].apply(family_d).T

per_family.round(3)

family,BigScience,EleutherAI,FLAN-T5,GLM-130B,LLaMA,OPT,OpenAI GPT
sentence_length_mean,0.179,0.702,0.165,-0.262,-0.172,0.457,-0.016
sentence_length_sd,-0.089,1.101,-0.238,-0.216,-0.027,0.433,-0.254
mattr,0.124,1.651,-0.641,-1.905,-1.212,1.232,-0.118
hapax_rate,0.215,1.439,-0.271,-2.315,-1.637,1.158,-0.316
stopword_rate,-0.313,-0.673,-0.070,0.570,0.411,-0.342,0.063
punctuation_rate,-0.195,-0.072,-0.082,0.462,0.229,-0.293,-0.232
noun_rate,0.207,0.089,0.217,-0.209,-0.093,-0.015,-0.025
verb_rate,0.097,0.130,0.024,0.391,0.254,0.173,0.116
past_tense_rate,-0.056,-0.081,0.018,0.014,-0.015,-0.082,-0.128
adverb_rate,-0.176,0.484,-0.321,-0.299,-0.320,0.247,-0.305


In [6]:
disagreement = pd.DataFrame({
    "d_pooled": separation["d_mage"],
    "d_lowest": per_family.min(axis=1),
    "d_highest": per_family.max(axis=1),
})
disagreement["range"] = disagreement["d_highest"] - disagreement["d_lowest"]
disagreement["positive_families"] = (per_family > 0).sum(axis=1)

print("features reaching d >= 0.5 for at least one family:")
print((per_family.abs() >= 0.5).any(axis=1).sum(), "of 14")
print("features where families disagree on the sign:")
print(((per_family > 0).any(axis=1) & (per_family < 0).any(axis=1)).sum(), "of 14")

disagreement.sort_values("range", ascending=False).round(3)

features reaching d >= 0.5 for at least one family:
6 of 14
features where families disagree on the sign:
13 of 14


,d_pooled,d_lowest,d_highest,range,positive_families
hapax_rate,-0.028,-2.315,1.439,3.755,3
mattr,0.009,-1.905,1.651,3.557,3
sentence_length_sd,0.133,-0.254,1.101,1.355,2
stopword_rate,-0.105,-0.673,0.570,1.243,3
sentence_length_mean,0.165,-0.262,0.702,0.965,4
flesch_reading_ease,-0.149,-0.528,0.379,0.906,2
adverb_rate,-0.068,-0.321,0.484,0.805,2
punctuation_rate,-0.091,-0.293,0.462,0.755,2
contraction_rate,0.088,-0.064,0.379,0.443,5
noun_rate,0.040,-0.209,0.217,0.426,3


## 3. Are the folds confounded with domain?

The folds in 04 hold one generator family out of training. If that family's texts are not spread evenly across the ten domains, holding it out changes the subject matter too, and a fall in performance cannot be attributed to the generator.

Each family's spread across the domains is compared with the overall machine mix by total variation distance (Gibbs and Su, 2002): 0 means identical, 1 means no overlap. A small distance means the fold changes the generator and little else.

In [7]:
machine = features[features["label"] == 1]

mix = pd.crosstab(machine["family"], machine["domain"], normalize="index")

# Total variation distance between each family's domain mix and the overall mix
# (Gibbs and Su, 2002): 0 means identical, 1 means no overlap
overall = machine["domain"].value_counts(normalize=True)
departure = (0.5 * (mix - overall).abs().sum(axis=1)).sort_values(ascending=False)

print("total variation distance from the overall machine mix:")
print(departure.round(3).to_string())
print()

mix.round(3)

total variation distance from the overall machine mix:
family
EleutherAI    0.049
BigScience    0.017
GLM-130B      0.011
FLAN-T5       0.009
OpenAI GPT    0.008
OPT           0.008
LLaMA         0.007



domain,cmv,eli5,hswag,roct,sci_gen,squad,tldr,wp,xsum,yelp
family,,,,,,,,,,
BigScience,0.085,0.107,0.107,0.108,0.098,0.105,0.084,0.108,0.110,0.089
EleutherAI,0.087,0.113,0.106,0.111,0.063,0.089,0.084,0.100,0.125,0.123
FLAN-T5,0.083,0.105,0.105,0.106,0.101,0.104,0.081,0.105,0.106,0.105
GLM-130B,0.086,0.107,0.106,0.108,0.087,0.107,0.082,0.105,0.107,0.105
LLaMA,0.084,0.105,0.105,0.106,0.100,0.104,0.082,0.104,0.107,0.105
OPT,0.086,0.108,0.102,0.106,0.098,0.101,0.083,0.099,0.110,0.108
OpenAI GPT,0.085,0.105,0.105,0.106,0.092,0.106,0.082,0.106,0.107,0.106


## 4. Save

Tables for Appendix D.

In [8]:
print("Table D.x - effect sizes on both corpora\n")
print(separation.sort_values("d_hc3", key=abs, ascending=False).round(3).to_markdown())

Table D.x - effect sizes on both corpora

|                      |   d_hc3 |   d_mage |   change |
|:---------------------|--------:|---------:|---------:|
| hapax_rate           |  -1.264 |   -0.028 |    1.236 |
| flesch_reading_ease  |  -0.812 |   -0.149 |    0.663 |
| sentence_length_mean |   0.801 |    0.165 |   -0.636 |
| adverb_rate          |  -0.776 |   -0.068 |    0.708 |
| punctuation_rate     |  -0.685 |   -0.091 |    0.595 |
| contraction_rate     |  -0.547 |    0.088 |    0.635 |
| pronoun_rate         |  -0.489 |    0.021 |    0.511 |
| first_person_rate    |  -0.44  |   -0.071 |    0.369 |
| sentence_length_sd   |   0.418 |    0.133 |   -0.285 |
| mattr                |  -0.341 |    0.009 |    0.35  |
| noun_rate            |   0.213 |    0.04  |   -0.174 |
| verb_rate            |  -0.151 |    0.142 |    0.293 |
| past_tense_rate      |  -0.094 |   -0.05  |    0.044 |
| stopword_rate        |   0.087 |   -0.105 |   -0.192 |


In [9]:
print("Table D.x - effect sizes per family, against the same human texts\n")
print(per_family.round(3).to_markdown())

Table D.x - effect sizes per family, against the same human texts

|                      |   BigScience |   EleutherAI |   FLAN-T5 |   GLM-130B |   LLaMA |    OPT |   OpenAI GPT |
|:---------------------|-------------:|-------------:|----------:|-----------:|--------:|-------:|-------------:|
| sentence_length_mean |        0.179 |        0.702 |     0.165 |     -0.262 |  -0.172 |  0.457 |       -0.016 |
| sentence_length_sd   |       -0.089 |        1.101 |    -0.238 |     -0.216 |  -0.027 |  0.433 |       -0.254 |
| mattr                |        0.124 |        1.651 |    -0.641 |     -1.905 |  -1.212 |  1.232 |       -0.118 |
| hapax_rate           |        0.215 |        1.439 |    -0.271 |     -2.315 |  -1.637 |  1.158 |       -0.316 |
| stopword_rate        |       -0.313 |       -0.673 |    -0.07  |      0.57  |   0.411 | -0.342 |        0.063 |
| punctuation_rate     |       -0.195 |       -0.072 |    -0.082 |      0.462 |   0.229 | -0.293 |       -0.232 |
| noun_rate          

## Observations

### No feature separates the classes on MAGE

Six features reached |d| of 0.5 or more on HC3. None do on MAGE. The largest is mean sentence length at 0.165. Hapax rate was the strongest on HC3 at -1.264 and is -0.028 here.

The order changed too. The Spearman correlation between the two rankings is 0.121, and five features change sign: contraction rate, pronoun rate, verb rate, stop-word rate and MATTR. The features were selected on HC3, so that selection did not transfer to the corpus the results are reported on. §3.6 records it; reselecting on MAGE would leak the evaluation corpus into the design.

### Because the families cancel

Measured per family against the same human pool, six of the fourteen reach |d| of 0.5 or more for at least one family, and thirteen of the fourteen have families on both sides of zero. Hapax rate runs from -2.315 on GLM-130B to +1.439 on EleutherAI, MATTR from -1.905 to +1.651. Verb rate is the only feature on which all seven families agree in sign.

Weighted by family size those extremes come to roughly the pooled values, so the near-zero figures above are cancellation rather than absence of signal. A model that learns MATTR's direction from six families applies the wrong sign to a seventh sitting on the other side of the human mean.

### The folds are not confounded with domain

Each family's domain mix has a total variation distance from the overall machine mix of about 0.05 and 0.01, and by about 0.02 for six of the seven. Holding a family out leaves the subject matter near enough unchanged, so a fall in those folds can be read as generator shift.

### Carried forward

The two tables in section 4 are reported in Appendix D.

Notebook 12 answers whether the 14 features affect the detector's output at all. They pass through one linear layer with no activation, so each feature's Shapley value is w_j(x_j - E[x_j]) exactly, and the eight seed-0 checkpoints give the answer without retraining.